# Model Experiments -- What Works and What Doesn't

This notebook walks through the ExoPred model iterations (v1 through v4), shows you what features drive predictions, and lets you experiment with different approaches.

**Key caveat:** All current models are trained on **calibration model labels**, not real measurements. The R-squared values look great (0.95+) but that's partly because the model is learning the calibration formula, not real biology. Your 80K dataset will be the real test.

In [ ]:
import os, sys, warnings, json
warnings.filterwarnings("ignore")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.exists("data"):
    os.chdir(os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import joblib

%matplotlib inline
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print(f"Working directory: {os.getcwd()}")

---
## 1. Model History: v1 through v4

| Version | Approach | Features | CV R2 | Leave-Seq-Out R2 | Notes |
|---------|----------|----------|-------|-------------------|-------|
| v1 (Phase 1) | PyTorch NN | ESM-2 + physicochemical | -- | -- | Abandoned (too complex for small data) |
| v2 | GBR | 72 (physchem + MEROPS + terminal + cell type) | 0.997 | 0.948 | Current production model |
| v3 | GBR + ESM-2 | v2 + ESM-2 embeddings | 0.918 | 0.918 | ESM-2 hurt (overfitting on 19 sequences) |
| v4 | GBR + Turk | v2 + Turk MMP Z-scores | 0.944 | 0.942 | Turk features didn't help for exopeptidase task |

**v2 is still the best.** The simpler feature set generalizes better with this small training set.

---
## 2. Feature Importance (v2 Model)

In [ ]:
fi = pd.read_csv("exopred/checkpoints/v2_feature_importance.csv")

# Top 20 features
top20 = fi.head(20).copy()

fig, ax = plt.subplots(figsize=(10, 8))
colors = []
for feat in top20["feature"]:
    if feat.startswith("cell_"):
        colors.append("#d32f2f")
    elif "protection" in feat or "mod" in feat:
        colors.append("#1976d2")
    elif feat.startswith("merops_"):
        colors.append("#43a047")
    else:
        colors.append("#ff9800")

ax.barh(top20["feature"][::-1], top20["importance"][::-1], color=colors[::-1])
ax.set_xlabel("Feature Importance (GBR)")
ax.set_title("ExoPred v2: Top 20 Features")

# Manual legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#d32f2f", label="Cell type"),
    Patch(facecolor="#1976d2", label="Terminal modification"),
    Patch(facecolor="#43a047", label="MEROPS cleavage frequency"),
    Patch(facecolor="#ff9800", label="Physicochemical"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.show()

print(f"cell_hMSC importance: {fi[fi['feature']=='cell_hMSC']['importance'].values[0]:.4f}")
print(f"That's {fi[fi['feature']=='cell_hMSC']['importance'].values[0] / fi['importance'].sum() * 100:.1f}% of total importance")
print(f"\nTop 5 MEROPS features:")
merops_feats = fi[fi["feature"].str.startswith("merops_")]
print(merops_feats.head(5).to_string(index=False))

### SAM'S EXPERTISE NEEDED

**Cell type dominates. In your experience, is hMSC really 4x more degradative than macrophage, or is that an artifact of the calibration model?**

The calibration model assigns aggressiveness values: hMSC=0.85 (mild), Macrophage=0.20 (aggressive). This means macrophages degrade peptides ~4x more. But the GBR learned that cell_hMSC is the #1 feature -- meaning the model mostly just learned to separate hMSC from everything else.

- Is the 4x difference real?
- Would the ranking change with your real measurements?
- Which cell type has the MOST variance in degradation (i.e., where sequence matters most)?

Notes: 

---
## 3. Leave-Sequence-Out Cross-Validation

This is the REAL test: can the model predict a peptide it has never seen? We hold out all 4 cell-type entries for one sequence at a time.

In [ ]:
# Rebuild the v2 training data
from exopred.train_v2 import (
    build_training_data, build_merops_cleavage_freq,
    compute_fraction_remaining, CELL_TYPES, N_MOD_BASE, C_MOD_BASE
)

X_df, y, feature_names = build_training_data()
X = X_df.values

# Build sequence groups for leave-sequence-out
rozans = pd.read_csv("data/rozans-618-enriched.csv")
paper1 = rozans[rozans["paper"] == "Paper 1 (ACS Biomater 2024)"].copy()
valid_n = set(N_MOD_BASE.keys())
valid_c = set(C_MOD_BASE.keys())
paper1 = paper1[paper1["n_terminal"].isin(valid_n) & paper1["c_terminal"].isin(valid_c)]

# Each peptide appears 4 times (once per cell type)
groups = np.repeat(np.arange(len(paper1)), len(CELL_TYPES))

print(f"Training data: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Unique sequences: {len(paper1)}, Groups: {len(np.unique(groups))}")

# Leave-one-group-out CV
logo = LeaveOneGroupOut()
gbr = GradientBoostingRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    min_samples_leaf=10, subsample=0.8, random_state=42,
)

y_pred_logo = cross_val_predict(gbr, X, y, groups=groups, cv=logo)

# Per-sequence error
seq_errors = []
for i, seq_idx in enumerate(np.unique(groups)):
    mask = groups == seq_idx
    y_true_seq = y[mask]
    y_pred_seq = y_pred_logo[mask]
    rmse = np.sqrt(mean_squared_error(y_true_seq, y_pred_seq))
    r2 = r2_score(y_true_seq, y_pred_seq) if len(y_true_seq) > 1 else np.nan
    seq_errors.append({
        "seq_idx": seq_idx,
        "sequence": paper1.iloc[seq_idx]["clean_sequence"],
        "n_terminal": paper1.iloc[seq_idx]["n_terminal"],
        "c_terminal": paper1.iloc[seq_idx]["c_terminal"],
        "rmse": rmse,
        "r2": r2,
        "mean_actual": y_true_seq.mean(),
        "mean_pred": y_pred_seq.mean(),
    })

seq_err_df = pd.DataFrame(seq_errors).sort_values("rmse", ascending=False)

overall_r2 = r2_score(y, y_pred_logo)
overall_rmse = np.sqrt(mean_squared_error(y, y_pred_logo))

print(f"\nLeave-Sequence-Out Results:")
print(f"  Overall R2:   {overall_r2:.4f}")
print(f"  Overall RMSE: {overall_rmse:.4f}")

# Predicted vs actual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y, y_pred_logo, alpha=0.3, s=20, color="steelblue")
axes[0].plot([0, 1], [0, 1], "r--", alpha=0.5)
axes[0].set_xlabel("Actual fraction_remaining_48h")
axes[0].set_ylabel("Predicted fraction_remaining_48h")
axes[0].set_title(f"Leave-Sequence-Out CV (R2={overall_r2:.4f})")

# Top 10 worst-predicted sequences
worst10 = seq_err_df.head(10)
axes[1].barh(worst10["sequence"][::-1], worst10["rmse"][::-1], color="#d32f2f")
axes[1].set_xlabel("RMSE")
axes[1].set_title("10 Hardest-to-Predict Sequences")
plt.tight_layout()
plt.show()

print(f"\n10 hardest sequences (highest RMSE):")
print(worst10[["sequence", "n_terminal", "c_terminal", "rmse", "mean_actual", "mean_pred"]].to_string(index=False))

### SAM'S EXPERTISE NEEDED

**The sequences above have the highest prediction error. What's different about them?**

Look at the 10 hardest-to-predict sequences. Ask yourself:
- Do they share a structural feature the model is missing?
- Are they the ones with unusual degradation behavior in your experiments?
- Do any of them have the Histidine exception or similar outlier behavior?

Notes on the hard cases: 

---
## 4. XGBoost Comparison

Does XGBoost do better than sklearn's GradientBoostingRegressor? (They use different algorithms under the hood.)

In [ ]:
try:
    from xgboost import XGBRegressor
    
    xgb = XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        min_child_weight=10, subsample=0.8, random_state=42,
        verbosity=0,
    )
    
    # Leave-sequence-out CV with XGBoost
    y_pred_xgb = cross_val_predict(xgb, X, y, groups=groups, cv=logo)
    xgb_r2 = r2_score(y, y_pred_xgb)
    xgb_rmse = np.sqrt(mean_squared_error(y, y_pred_xgb))
    
    print(f"XGBoost Leave-Sequence-Out:")
    print(f"  R2:   {xgb_r2:.4f} (vs GBR {overall_r2:.4f})")
    print(f"  RMSE: {xgb_rmse:.4f} (vs GBR {overall_rmse:.4f})")
    
    if xgb_r2 > overall_r2:
        print(f"\n  XGBoost wins by {xgb_r2 - overall_r2:.4f} R2")
    else:
        print(f"\n  GBR wins by {overall_r2 - xgb_r2:.4f} R2")
    
    # Compare predictions
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_pred_logo, y_pred_xgb, alpha=0.3, s=20, color="steelblue")
    ax.plot([0, 1], [0, 1], "r--", alpha=0.5)
    ax.set_xlabel("GBR Predictions")
    ax.set_ylabel("XGBoost Predictions")
    ax.set_title("GBR vs XGBoost (Leave-Seq-Out)")
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print("Then re-run this cell to compare.")

---
## 5. Feature Subset Ablation

Which features actually generalize? Let's try: (a) only terminal features, (b) only MEROPS features, (c) only physicochemical. The subset with the best leave-seq-out R2 tells you what the model is really learning.

In [ ]:
# Define feature subsets
terminal_cols = [c for c in feature_names if "protection" in c or "mod" in c]
cell_cols = [c for c in feature_names if c.startswith("cell_")]
merops_cols = [c for c in feature_names if c.startswith("merops_")]
physchem_cols = [c for c in feature_names if c not in terminal_cols + cell_cols + merops_cols]

subsets = {
    "Terminal + Cell only": terminal_cols + cell_cols,
    "MEROPS + Cell only": merops_cols + cell_cols,
    "Physicochemical + Cell only": physchem_cols + cell_cols,
    "Terminal + MEROPS + Cell (no physchem)": terminal_cols + merops_cols + cell_cols,
    "Everything (v2 baseline)": feature_names,
}

results = {}
for name, cols in subsets.items():
    col_idx = [feature_names.index(c) for c in cols]
    X_sub = X[:, col_idx]
    
    gbr_sub = GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        min_samples_leaf=10, subsample=0.8, random_state=42,
    )
    
    y_pred_sub = cross_val_predict(gbr_sub, X_sub, y, groups=groups, cv=logo)
    r2 = r2_score(y, y_pred_sub)
    rmse = np.sqrt(mean_squared_error(y, y_pred_sub))
    
    results[name] = {"r2": r2, "rmse": rmse, "n_features": len(cols)}
    print(f"{name:45s}  R2={r2:.4f}  RMSE={rmse:.4f}  ({len(cols)} features)")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
r2s = [results[n]["r2"] for n in names]
colors = ["#43a047" if r == max(r2s) else "#1976d2" for r in r2s]
ax.barh(names[::-1], r2s[::-1], color=colors[::-1])
ax.set_xlabel("Leave-Sequence-Out R2")
ax.set_title("Feature Subset Ablation: What Generalizes?")
ax.axvline(r2s[-1], color="gray", linestyle="--", alpha=0.5, label="Full model")
plt.tight_layout()
plt.show()

best = max(results.items(), key=lambda x: x[1]["r2"])
print(f"\nBest subset: {best[0]} (R2={best[1]['r2']:.4f})")
print(f"Interpretation: The model's generalization is mostly driven by these {best[1]['n_features']} features.")

---
## 6. SHAP Analysis (Feature Contributions per Prediction)

SHAP values show which features push each prediction UP (more stable) or DOWN (more degradation) for specific peptides.

In [ ]:
try:
    import shap
    
    # Train final model on all data for SHAP
    gbr_final = GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        min_samples_leaf=10, subsample=0.8, random_state=42,
    )
    gbr_final.fit(X, y)
    
    # SHAP values
    explainer = shap.TreeExplainer(gbr_final)
    shap_values = explainer.shap_values(X)
    
    # Summary plot (beeswarm)
    fig, ax = plt.subplots(figsize=(10, 10))
    shap.summary_plot(shap_values, X_df, feature_names=feature_names, 
                      max_display=20, show=False)
    plt.title("SHAP Summary: Top 20 Features")
    plt.tight_layout()
    plt.show()
    
    # For a specific peptide: show SHAP waterfall
    # Pick the most-stable and least-stable predictions
    most_stable_idx = np.argmax(y)
    least_stable_idx = np.argmin(y)
    
    print(f"\nMost stable sample (index {most_stable_idx}): predicted={gbr_final.predict(X[most_stable_idx:most_stable_idx+1])[0]:.3f}")
    print(f"Least stable sample (index {least_stable_idx}): predicted={gbr_final.predict(X[least_stable_idx:least_stable_idx+1])[0]:.3f}")
    
    print("\n(For individual SHAP waterfall plots, run: shap.plots.waterfall(shap.Explanation(...))")
    
except ImportError:
    print("SHAP not installed. Run: pip install shap")
    print()
    print("What SHAP would show you:")
    print("- For each prediction, which features pushed it UP (toward stable) vs DOWN (toward degraded)")
    print("- For the Histidine peptides: whether MEROPS features or physicochemical features drive the low stability prediction")
    print("- Whether cell_type features dominate individual predictions as much as they dominate global importance")
    print()
    print("This is the most actionable analysis for understanding model behavior.")

---
## 7. Placeholder: Train on Your 80K Data

When you add your real measurements, this cell will retrain the v2 model and show whether R2 improves with real labels vs calibration model labels.

In [ ]:
# ============================================================
# ADD YOUR 80K DATA HERE
# ============================================================
YOUR_DATA_PATH = None  # e.g., "data/rozans-80k-measured.csv"
SEQUENCE_COL = "sequence"  # column name for peptide sequence
TARGET_COL = "fraction_remaining_48h"  # column name for the measurement

if YOUR_DATA_PATH is not None and os.path.exists(YOUR_DATA_PATH):
    from exopred.train_v2 import (
        physicochemical_features, merops_features, terminal_mod_features,
        cell_type_features, featurize_one
    )
    
    your_data = pd.read_csv(YOUR_DATA_PATH)
    print(f"Loaded {len(your_data)} rows from {YOUR_DATA_PATH}")
    
    # Build features (you may need to adjust column names)
    freq_tables = build_merops_cleavage_freq()
    
    rows = []
    for _, row in your_data.iterrows():
        seq = str(row[SEQUENCE_COL])
        n_mod = str(row.get("n_terminal", "NH2"))
        c_mod = str(row.get("c_terminal", "COOH"))
        cell = str(row.get("cell_type", "hUVEC"))
        feats = featurize_one(seq, n_mod, c_mod, cell, freq_tables)
        rows.append(feats)
    
    X_new = pd.DataFrame(rows)
    y_new = your_data[TARGET_COL].values
    
    # Ensure feature alignment
    for col in feature_names:
        if col not in X_new.columns:
            X_new[col] = 0.0
    X_new = X_new[feature_names]
    
    # Build groups for leave-seq-out
    seq_labels = pd.factorize(your_data[SEQUENCE_COL])[0]
    
    # Train and evaluate
    gbr_new = GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        min_samples_leaf=10, subsample=0.8, random_state=42,
    )
    
    y_pred_new = cross_val_predict(gbr_new, X_new.values, y_new, 
                                    groups=seq_labels, cv=LeaveOneGroupOut())
    new_r2 = r2_score(y_new, y_pred_new)
    new_rmse = np.sqrt(mean_squared_error(y_new, y_pred_new))
    
    print(f"\nResults with YOUR data:")
    print(f"  Leave-Seq-Out R2:   {new_r2:.4f} (vs calibration model: {overall_r2:.4f})")
    print(f"  Leave-Seq-Out RMSE: {new_rmse:.4f} (vs calibration model: {overall_rmse:.4f})")
    
    if new_r2 > overall_r2:
        print(f"\n  IMPROVEMENT: +{new_r2 - overall_r2:.4f} R2 with real measurements!")
    else:
        print(f"\n  Real data is harder to predict (as expected). The model is now learning")
        print(f"  real biology instead of a calibration formula.")
    
    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_new, y_pred_new, alpha=0.3, s=20, color="steelblue")
    ax.plot([y_new.min(), y_new.max()], [y_new.min(), y_new.max()], "r--", alpha=0.5)
    ax.set_xlabel("Actual (your measurements)")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Your 80K Data: Leave-Seq-Out (R2={new_r2:.4f})")
    plt.tight_layout()
    plt.show()
    
else:
    print("No data path set yet. Update YOUR_DATA_PATH, SEQUENCE_COL, and TARGET_COL above.")
    print()
    print("Expected format:")
    print("  - CSV with columns: sequence, n_terminal, c_terminal, cell_type, fraction_remaining_48h")
    print("  - Or at minimum: sequence and fraction_remaining_48h (defaults will be used for mods/cell)")
    print()
    print("When you run this with real data, the key question is:")
    print("  Does R2 go UP (real data confirms the model) or DOWN (real data reveals the model was")
    print("  just memorizing the calibration formula)?")

---
## 8. Bottger External Validation

The acid test: can the model trained on your calibration data predict stability for completely different peptides (antimicrobial) in completely different conditions (human plasma/serum)?

**Spoiler: correlation is near zero.** This section explains why and what data would fix it.

In [ ]:
from exopred.train_v2 import featurize_one

bottger = pd.read_csv("data/external_validation/extracted_data.csv")
freq_tables = build_merops_cleavage_freq()

# Load the trained v2 model
bundle = joblib.load("exopred/checkpoints/exopred_v2_gbr.pkl")
gbr_v2 = bundle["gbr"]
v2_features = bundle["feature_names"]

# Filter to entries with usable data
bott_valid = bottger[bottger["pct_intact"].notna() & bottger["sequence"].notna()].copy()

# Some sequences have non-standard characters -- filter
STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")
bott_valid = bott_valid[bott_valid["sequence"].apply(
    lambda s: all(c in STANDARD_AAS for c in str(s).upper())
)]

print(f"Bottger entries with valid data: {len(bott_valid)}")

# Predict stability for each Bottger peptide
# Map Bottger conditions to our feature space (approximate)
preds = []
for _, row in bott_valid.iterrows():
    seq = str(row["sequence"]).upper()
    # Bottger peptides: assume unprotected termini unless noted
    mods = str(row.get("modification_notes", ""))
    n_mod = "Ac" if "acetyl" in mods.lower() or "ac-" in mods.lower() else "NH2"
    c_mod = "amide" if "amide" in mods.lower() or "-nh2" in mods.lower() else "COOH"
    
    feats = featurize_one(seq, n_mod, c_mod, "hUVEC", freq_tables)
    feats_df = pd.DataFrame([feats])
    
    for col in v2_features:
        if col not in feats_df.columns:
            feats_df[col] = 0.0
    feats_df = feats_df[v2_features]
    
    pred = gbr_v2.predict(feats_df.values)[0]
    preds.append(pred)

bott_valid = bott_valid.copy()
bott_valid["predicted_stability"] = preds
bott_valid["actual_stability"] = bott_valid["pct_intact"] / 100.0  # Normalize to 0-1

# Correlation
from scipy import stats
rho, p_val = stats.spearmanr(bott_valid["predicted_stability"], bott_valid["actual_stability"])

print(f"\nSpearman correlation: {rho:.4f} (p={p_val:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(bott_valid["actual_stability"], bott_valid["predicted_stability"], 
                alpha=0.6, s=40, color="steelblue")
axes[0].plot([0, 1], [0, 1], "r--", alpha=0.5)
axes[0].set_xlabel("Actual % Intact (Bottger)")
axes[0].set_ylabel("Predicted Stability (ExoPred v2)")
axes[0].set_title(f"Bottger External Validation\nSpearman rho={rho:.3f}")

# Per-peptide comparison
pep_avg = bott_valid.groupby("peptide_name").agg(
    actual=("actual_stability", "mean"),
    predicted=("predicted_stability", "mean"),
).sort_values("actual", ascending=False)

x = range(len(pep_avg))
axes[1].bar([i - 0.2 for i in x], pep_avg["actual"], 0.4, label="Actual", color="#1976d2", alpha=0.8)
axes[1].bar([i + 0.2 for i in x], pep_avg["predicted"], 0.4, label="Predicted", color="#d32f2f", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(pep_avg.index, rotation=90, fontsize=7)
axes[1].set_ylabel("Stability (0-1)")
axes[1].set_title("Per-Peptide: Actual vs Predicted")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nWhy is correlation ~0?")
print(f"1. Training data: cell culture conditions (hMSC/hUVEC/macrophage/THP-1)")
print(f"   Validation data: human plasma/blood/serum (different protease milieu)")
print(f"2. Training labels: computed from calibration model, not real measurements")
print(f"3. Peptide domain: training = adhesion peptides, validation = antimicrobial")
print(f"4. Sequence lengths: training = 6-7 AA, Bottger = 10-20+ AA")
print(f"\nWhat would fix it:")
print(f"- Your 80K real measurements (same domain, real labels)")
print(f"- PEPlife2 plasma half-lives as additional training data")
print(f"- Broader sequence length coverage in training")